# featherweight-ai — Kaggle smoke test (Milestone C)

Deliberately trivial. Proves the local → GitHub → Kaggle bridge works **before**
anything depends on it, and turns the project's biggest assumption (no bf16 on
free Kaggle) from analysis into measurement.

**Before running — right sidebar:**

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** |
| Internet | **On** |
| Add-ons → Secrets | `HF_TOKEN` = your Hugging Face read token |

P100 is *not* the choice: sm_60 has no fp16 tensor cores and `LLM.int8()`
requires sm_75+. See `docs/memory.md` D6.

## 1. What hardware did we actually get?

In [ ]:
!nvidia-smi

## 2. The bf16 question

The whole project rests on this. `is_bf16_supported()` called plainly can return
`True` on a T4 because PyTorch will **emulate** bf16 via fp32 — which works, is
slow, and gives none of the benefit. The two lines below separate what the
silicon can do from what the software will pretend to do.

bf16 tensor cores arrived with **Ampere (sm_80)**. Turing is **sm_75**.

In [ ]:
import torch

print("torch    :", torch.__version__, "| CUDA:", torch.version.cuda)
print("devices  :", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} = {p.name}, capability=({p.major}, {p.minor}), "
          f"{p.total_memory / 1024**3:.1f} GiB")

cap = torch.cuda.get_device_capability(0)
print("\n--- bf16 ---")
print("is_bf16_supported()                          =", torch.cuda.is_bf16_supported())
try:
    print("is_bf16_supported(including_emulation=False) =",
          torch.cuda.is_bf16_supported(including_emulation=False))
except TypeError:
    print("is_bf16_supported(including_emulation=False) =  <arg not in this torch>")
print("native, i.e. capability >= (8, 0)            =", cap >= (8, 0))

## 3. Read the HF token from Secrets

Never hardcode a token in a cell — Kaggle notebooks are public by default and
a committed token is a leaked token. This prints only the length, never the value.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = token          # picked up automatically by huggingface_hub
print("HF_TOKEN retrieved OK, length:", len(token))

## 4. The actual test: clone the repo and run `check_env.py` unmodified

This is the bridge. The same script that passes on the RTX 3060 should run here
with **zero changes** and report different — correct — hardware facts.

In [ ]:
!rm -rf /kaggle/working/featherweight-ai
!git clone -q https://github.com/AadiPathak23/featherweight-ai.git /kaggle/working/featherweight-ai
!ls /kaggle/working/featherweight-ai

In [ ]:
# check_env.py needs bitsandbytes; Kaggle's base image may not have it.
!pip install -q bitsandbytes

In [ ]:
!cd /kaggle/working/featherweight-ai && python scripts/check_env.py

## 5. Success criteria

- [ ] `nvidia-smi` shows **two** Tesla T4s
- [ ] `capability = (7, 5)`
- [ ] `bf16_native = False` ← the finding this milestone exists for
- [ ] `HF_TOKEN` read from Secrets
- [ ] `check_env.py` runs unmodified from the cloned repo

**Then:** find the GPU-hour meter (Settings, or the session panel) so the
~30 hr/week budget is visible from day one, and **stop the session explicitly**
when done — an idle session burns quota.